# Single-cell image example (scprotrait)

In [ ]:
import annbatch
import anndata as ad

In [1]:
from pathlib import Path

H5AD_PATH = Path("/vol/data/annbatch/scportrait/input_data_Golgi")
COLLECTION_PATH = Path("/vol/data/annbatch/scportrait.zarr")

## Create annbatch collection

In [3]:
import scipy.sparse as sp
import h5py


def load_adata(path):
    chunk_size = 128
    
    f = h5py.File(path, "r")
    obs = ad.io.read_elem(f["obs"])
    var = ad.io.read_elem(f["var"])
    obsm = {
        k: ad.experimental.read_elem_lazy(
            f["obsm"][k], 
            chunks=(chunk_size,) + (-1,) * (f["obsm"][k].ndim - 1)
        )
        for k in f["obsm"]
    }
    uns = ad.io.read_elem(f["uns"]) if "uns" in f else {}
    return ad.AnnData(
        X=sp.csr_matrix((len(obs), len(var))),
        obs=obs,
        var=var,
        obsm=obsm,
        uns=uns,
    )


In [4]:
collection = annbatch.DatasetCollection(COLLECTION_PATH)
collection.add_adatas(
    adata_paths=list(H5AD_PATH.glob("*.h5sc")),
    load_adata=load_adata,
    zarr_dense_chunk_size=64,
    zarr_dense_shard_size=32768,
    n_obs_per_dataset=262144,
    shuffle_chunk_size=128,
)

checking for mismatched keys:   0%|          | 0/5 [00:00<?, ?it/s]

loading:   0%|          | 0/5 [00:00<?, ?it/s]

/vol/data/miniconda3/envs/annbatch/lib/python3.12/site-packages/anndata/_core/anndata.py:1823: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


processing chunks:   0%|          | 0/14 [00:00<?, ?it/s]

/vol/data/miniconda3/envs/annbatch/lib/python3.12/site-packages/zarr/api/asynchronous.py:247: ZarrUserWarning: Consolidated metadata is currently not part in the Zarr format 3 specification. It may not be supported by other zarr implementations and may change in the future.
  warnings.warn(
/vol/data/miniconda3/envs/annbatch/lib/python3.12/site-packages/zarr/api/asynchronous.py:247: ZarrUserWarning: Consolidated metadata is currently not part in the Zarr format 3 specification. It may not be supported by other zarr implementations and may change in the future.
  warnings.warn(
/vol/data/miniconda3/envs/annbatch/lib/python3.12/site-packages/zarr/api/asynchronous.py:247: ZarrUserWarning: Consolidated metadata is currently not part in the Zarr format 3 specification. It may not be supported by other zarr implementations and may change in the future.
  warnings.warn(
/vol/data/miniconda3/envs/annbatch/lib/python3.12/site-packages/zarr/api/asynchronous.py:247: ZarrUserWarning: Consolidated 

## Benchmark utils

In [3]:
import time

from tqdm import tqdm


def benchmark_loader(loader, n_samples, batch_size):
    num_iter = n_samples // batch_size
    loader_iter = loader.__iter__()

    start_time = time.time()
    batch_times = []
    batch_time = time.time()
    total_time = time.time()
    for i, _batch in tqdm(enumerate(loader_iter), total=num_iter):
        batch_times.append(time.time() - batch_time)
        batch_time = time.time()
        if i == num_iter:
            break

    total_time = time.time() - total_time

    execution_time = time.time() - start_time
    time_per_sample = (1e6 * execution_time) / (num_iter * batch_size)
    print(f"time per sample: {time_per_sample:.2f} μs")
    samples_per_sec = num_iter * batch_size / execution_time
    print(f"samples per sec: {samples_per_sec:.2f} samples/sec")

    return samples_per_sec, time_per_sample, batch_times, total_time

## Benchmark annbatch

In [4]:
import zarr
import zarrs

zarr.config.set(
    {"codec_pipeline.path": "zarrs.ZarrsCodecPipeline"}
)

In [7]:
import anndata as ad

from annbatch import Loader

In [8]:
f = zarr.open(COLLECTION_PATH)

obs =[ad.io.read_elem(f[f"dataset_{i}"]["obs"]) for i in range(len(f))]
var = [ad.io.read_elem(f[f"dataset_{i}"]["var"]) for i in range(len(f))]
datasets = [f[f"dataset_{i}"]["obsm"]["single_cell_images"] for i in range(len(f))]

In [9]:
BATCH_SIZE = 4096

loader = Loader(
    chunk_size=128,
    preload_nchunks=128,
    shuffle=True,
    batch_size=BATCH_SIZE,
    preload_to_gpu=True,
    to_torch=True,
    concat_strategy="shuffle-concat",
)
loader.add_datasets(
    datasets=datasets,
    obs=obs,
    var=var,
)

In [10]:
_ = benchmark_loader(loader, 200_000, BATCH_SIZE)

100%|██████████| 48/48 [00:15<00:00,  3.03it/s]

time per sample: 80.63 μs
samples per sec: 12402.84 samples/sec


## Benchmark scprotrait

In [4]:
from scportrait.tools.ml.datasets import HDF5SingleCellDataset

/vol/data/miniconda3/envs/scportrait/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/vol/data/miniconda3/envs/scportrait/lib/python3.12/site-packages/xarray_schema/__init__.py:1: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import DistributionNotFound, get_distribution


In [6]:
from scportrait.tools.ml.datasets import H5ScSingleCellDataset


dataset = H5ScSingleCellDataset(
    [str(p) for p in H5AD_PATH.glob("*.h5sc")],
    [0, 1, 2, 3, 4],
    transform=None,
    return_id=True
)

Total single cell records: 3621635


In [7]:
from torch.utils.data import DataLoader

BATCH_SIZE = 4096

dataloader = DataLoader(dataset, num_workers=8, batch_size=BATCH_SIZE, shuffle=True)

In [8]:
_ = benchmark_loader(dataloader, 200_000, BATCH_SIZE)

100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 48/48 [06:00<00:00,  7.52s/it]


time per sample: 1835.24 μs
samples per sec: 544.89 samples/sec
